# EnerGIS Framework - Runner

Haupteinstiegspunkt für Optimierungsläufe mit dem EnerGIS Planning Framework.

## Übersicht

Dieses Notebook führt einen vollständigen Optimierungslauf durch:
- **Perfect Forecast (PF)**: Optimale Dimensionierung über den gesamten Zeitraum
- **Rolling Horizon (RH)**: Operative Planung mit rollendem Horizont
- **PF → RH**: Kombinierter Workflow mit Design-Fixierung

## Quick Start

1. Alle Zellen mit **Run All** ausführen
2. Bei Bedarf Config-Pfade in Zelle 2 anpassen
3. Ergebnisse werden in `exports/` gespeichert

---

## 1. Setup & Imports

In [ ]:
# Auto-Setup: Projekt-Root finden und zum Path hinzufügen
from pathlib import Path
import sys
import os

def find_project_root(start: Path) -> Path:
    """Findet das Projekt-Root-Verzeichnis."""
    for candidate in [start] + list(start.parents):
        if (candidate / '.git').exists() and (candidate / 'energis').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Projekt-Root: {PROJECT_ROOT}")

In [ ]:
# Imports
import warnings
from datetime import datetime

from energis.run import rolling_horizon as rh
from energis.run import orchestrator

warnings.filterwarnings('ignore')
print("✅ Imports erfolgreich")

## 2. Konfiguration

Die Konfiguration erfolgt über YAML-Dateien, die in der angegebenen Reihenfolge gemerged werden.
Spätere Dateien überschreiben frühere Einträge.

### Standard-Konfiguration:
- `base.yaml` - Basis-Einstellungen (Solver, Zeitschritt, etc.)
- `tech_catalog.yaml` - Technologie-Katalog (Komponenten-Definitionen)
- `default.site.yaml` - Standort-Daten (Input-Daten, Zeitzone, etc.)
- `baseline.system.yaml` - System-Topologie (Komponenten, Kapazitäten)
- `pf_then_rh.workflow.scenario.yaml` - Szenario (Run-Mode, RH-Parameter)

Passe die Config-Pfade nach Bedarf an!

In [ ]:
# Konfigurationsdateien
CONFIG_PATHS = [
    'configs/base.yaml',
    'configs/tech_catalog.yaml',
    'configs/sites/default.site.yaml',
    'configs/systems/baseline.system.yaml',
    'configs/scenarios/pf_then_rh.workflow.scenario.yaml',
]

# Optional: Overrides für spezifische Parameter
# Beispiele:
# - Run-Mode ändern: {'scenario': {'run_mode': 'PF_ONLY'}}
# - Solver ändern: {'run': {'solver': 'glpk'}}
# - RH-Parameter: {'scenario': {'rolling_horizon': {'heat_horizon_hours': 72}}}
OVERRIDES = None

# Config-Dateien prüfen
print("📋 Konfigurationsdateien:")
all_exist = True
for cfg_path in CONFIG_PATHS:
    full_path = PROJECT_ROOT / cfg_path
    exists = full_path.exists()
    symbol = '✅' if exists else '❌'
    print(f"  {symbol} {cfg_path}")
    if not exists:
        all_exist = False

if not all_exist:
    raise FileNotFoundError("Nicht alle Config-Dateien gefunden!")

print("\n✅ Konfiguration OK")

## 3. Workflow ausführen

Der Workflow führt die Optimierung gemäß der konfigurierten Run-Mode aus:
- **PF_ONLY**: Nur Perfect Forecast
- **RH_ONLY**: Nur Rolling Horizon
- **PF_THEN_RH**: PF für Dimensionierung, dann RH mit fixiertem Design

In [ ]:
%%time
print("="*70)
print("🚀 STARTE OPTIMIERUNG")
print("="*70)
print(f"Start: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

try:
    workflow = rh.run_workflow(CONFIG_PATHS, overrides=OVERRIDES)
    
    print("\n" + "="*70)
    print("✅ OPTIMIERUNG ERFOLGREICH")
    print("="*70)
    print(f"\n📊 Workflow: {' → '.join(workflow.plan.steps)}")
    
    optimization_success = True
    
except Exception as e:
    print("\n" + "="*70)
    print("❌ FEHLER")
    print("="*70)
    print(f"\nFehler: {e}\n")
    
    import traceback
    traceback.print_exc()
    
    workflow = None
    optimization_success = False

## 4. Ergebnisse

Zeigt die wichtigsten Kennzahlen aus dem Optimierungslauf.

In [ ]:
if optimization_success and workflow:
    print("\n" + "="*70)
    print("📊 ERGEBNISSE")
    print("="*70)
    
    # Perfect Forecast Ergebnisse
    if workflow.pf_result:
        print("\n🎯 Perfect Forecast (PF):")
        print(f"  Zeitschritte:  {len(workflow.pf_result.table)}")
        
        if workflow.pf_result.costs:
            obj_value = workflow.pf_result.costs.get('objective.OBJ_value_EUR')
            if obj_value is not None:
                print(f"  Gesamtkosten:  {obj_value:,.0f} EUR")
            
            peak_power = workflow.pf_result.costs.get('P_buy_peak_MW')
            if peak_power is not None:
                print(f"  Peak-Leistung: {peak_power:.2f} MW")
    
    # Rolling Horizon Ergebnisse
    if workflow.rh_result:
        print("\n🔄 Rolling Horizon (RH):")
        print(f"  Fenster:       {len(workflow.rh_result.windows)}")
        print(f"  Zeitschritte:  {len(workflow.rh_result.table)}")
        
        if workflow.rh_result.costs:
            obj_value = workflow.rh_result.costs.get('objective.OBJ_value_EUR')
            if obj_value is not None:
                print(f"  Gesamtkosten:  {obj_value:,.0f} EUR")
    
    # Design
    if workflow.design:
        print("\n🏭 Anlagen-Design:")
        
        if workflow.design.heat_pumps:
            print("  Wärmepumpen:")
            for hp_id, hp_data in sorted(workflow.design.heat_pumps.items()):
                capacity = hp_data.get('capacity_mw', 0.0)
                print(f"    {hp_id}: {capacity:.2f} MW")
        
        if workflow.design.storage:
            storage_capacity = workflow.design.storage.get('capacity_mwh', 0.0)
            print(f"  Speicher:      {storage_capacity:.2f} MWh")
    
    print("\n" + "="*70)
else:
    print("⚠️  Keine Ergebnisse verfügbar")

In [ ]:
if optimization_success and workflow:
    from energis.io.plotter import export_plots, HAVE_MATPLOTLIB
    import json
    
    # Bestimme welche Daten für Plots verwendet werden
    plot_result = workflow.rh_result if workflow.rh_result else workflow.pf_result
    
    if plot_result:
        # Export-Verzeichnis erstellen
        export_dir = PROJECT_ROOT / 'notebooks' / 'exports' / 'latest_run'
        export_dir.mkdir(parents=True, exist_ok=True)
        
        # === PLOTS ERSTELLEN ===
        if HAVE_MATPLOTLIB:
            print("\n📊 Erstelle Visualisierungen...")
            
            try:
                # Summary für cost breakdown
                summary_sections = None
                if hasattr(plot_result, 'summary'):
                    summary_sections = plot_result.summary
                elif hasattr(plot_result, 'costs'):
                    summary_sections = {'objective': plot_result.costs}
                
                plot_files = export_plots(
                    str(export_dir),
                    plot_result.table,
                    plot_result.series,
                    summary_sections=summary_sections,
                    dpi=200
                )
                
                if plot_files:
                    print(f"✅ {len(plot_files)} Plot(s) erstellt:")
                    for pf in plot_files:
                        print(f"   📈 {pf}")
                else:
                    print("⚠️  Keine Plots erstellt (möglicherweise fehlende Daten)")
            except Exception as e:
                print(f"⚠️  Fehler beim Erstellen der Plots: {e}")
        else:
            print("⚠️  Matplotlib nicht verfügbar - Plots werden übersprungen")
        
        # === DATEN EXPORTIEREN ===
        print("\n💾 Exportiere Daten...")
        
        try:
            import pandas as pd
            
            # Zeitreihen als CSV exportieren
            ts_df = pd.DataFrame(plot_result.series, index=plot_result.table.index)
            csv_path = export_dir / 'timeseries.csv'
            ts_df.to_csv(csv_path)
            print(f"✅ Zeitreihen: {csv_path}")
            
            # Kosten als JSON exportieren
            if hasattr(plot_result, 'costs') and plot_result.costs:
                costs_path = export_dir / 'costs.json'
                with open(costs_path, 'w', encoding='utf-8') as f:
                    json.dump(plot_result.costs, f, indent=2, ensure_ascii=False)
                print(f"✅ Kosten: {costs_path}")
            
            # Design als JSON exportieren
            if workflow.design:
                design_dict = {
                    'heat_pumps': workflow.design.heat_pumps,
                    'storage': workflow.design.storage
                }
                design_path = export_dir / 'design.json'
                with open(design_path, 'w', encoding='utf-8') as f:
                    json.dump(design_dict, f, indent=2, ensure_ascii=False)
                print(f"✅ Design: {design_path}")
            
            print(f"\n📁 Alle Dateien in: {export_dir}")
            
        except Exception as e:
            print(f"⚠️  Fehler beim Exportieren: {e}")
            import traceback
            traceback.print_exc()
else:
    print("⚠️  Keine Ergebnisse zum Exportieren verfügbar")

## 4a. Visualisierung & Datenexport

Erstellt automatisch Plots (Wärmebilanz, elektrische Bilanz, Speicher, Kostenaufschlüsselung) und exportiert Ergebnisse als CSV/JSON.

## 5. Export (Optional)

Vollständiger Export aller Ergebnisse als Excel/CSV/JSON in `exports/`.

In [ ]:
# Uncomment to export results
# if optimization_success:
#     print("📦 Exportiere Ergebnisse...")
#     export_meta = orchestrator.run_all(CONFIG_PATHS, overrides=OVERRIDES)
#     
#     print(f"\n✅ Export abgeschlossen:")
#     print(f"  Verzeichnis: {export_meta['outdir']}")
#     print(f"  Excel:       {export_meta.get('scenario_xlsx')}")
#     print(f"  Design-JSON: {export_meta.get('pf_design_json')}")

## 6. Detaillierte Analyse (Optional)

Für detaillierte Analysen und Visualisierungen siehe:
- `scenario_studio.ipynb` - Interaktives Dashboard mit Plots und KPIs
- `synthetic_example.ipynb` - Beispiel mit synthetischen Daten
- `validation.ipynb` - Validierung gegen Referenz-Daten

---

## Weitere Informationen

- **Dokumentation**: `README.md`, `ARCHITECTURE_V2.md`
- **Methodologie**: `docs/methodology.md`
- **CLI-Nutzung**: `python -m energis.run.rolling_horizon --help`